# TEXBAT — universal features v4: geolocation + model-ready tables

Ta wersja notebooka rozszerza v3 o dane geolokalizacyjne z `navsol.mat`.

Nowe kolumny z `navsol.mat`:

- `ecef_x_m`, `ecef_y_m`, `ecef_z_m`
- `latitude_deg`, `longitude_deg`, `height_m`
- `latitude_delta_deg`, `longitude_delta_deg`, `height_delta_m`
- `distance_from_start_m`
- `position_delta_m`
- `speed_mps`
- `clock_error_m`
- `clock_error_rate_mps`
- `solution_flag`
- `nis_ratio`

Główne pliki wyjściowe:

```text
texbat_universal_features_v4/event_features_v4.csv
texbat_universal_features_v4/satellite_second_features_v4.csv
texbat_universal_features_v4/global_time_features_v4.csv
texbat_universal_features_v4/model_ready_satellite_second_features_v4.csv
texbat_universal_features_v4/model_ready_global_time_features_v4.csv
```

Najbardziej polecany plik do pierwszego modelu:

```text
model_ready_global_time_features_v4.csv
```

Uwaga: `latitude_deg` i `longitude_deg` są dobre do wizualizacji, ale do modelu bezpieczniejsze są cechy względne:
`distance_from_start_m`, `position_delta_m`, `height_delta_m`, `clock_error_m`, `clock_error_rate_mps`.

## 1. Imports and configuration

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

# =========================
# CONFIG
# =========================

BASE_DIR = Path("processed")
SCENARIOS = ["cleanStatic", "ds2", "ds3", "ds7"]

OUTPUT_DIR = Path("texbat_universal_features_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1-second bins, robust for merging channel/iq/iqtaps/navsol.
TIME_BIN_DECIMALS = 0

# Large files.
LOAD_CORRELATOR_FEATURES = True
LOAD_DYNAMIC_IQ_PSD_FEATURES = True

# Limits after file is loaded to RAM.
IQTAPS_MAX_ROWS = 300_000
IQ_MAX_ROWS = None

# PSD-like features from iq.mat require enough samples per bin.
PSD_MIN_SAMPLES = 8

# Global power columns are saved for analysis, but not recommended for first model.
KEEP_GLOBAL_POWER_FOR_ANALYSIS = True

print("BASE_DIR:", BASE_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("SCENARIOS:", SCENARIOS)

BASE_DIR: /home/aw/kosciuszkon/processed
OUTPUT_DIR: /home/aw/kosciuszkon/texbat_universal_features_v4
SCENARIOS: ['cleanStatic', 'ds2', 'ds3', 'ds7']


## 2. Helpers

In [3]:
def load_grid_mat(path: Path, variable_name: str | None = None, transpose: bool = True) -> np.ndarray:
    """Load TEXBAT/GRID .mat file.

    GRID docs say matrices should be transposed after loading.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    mat = sio.loadmat(path)
    keys = [k for k in mat.keys() if not k.startswith("__")]

    if variable_name is None:
        if len(keys) != 1:
            raise ValueError(f"More than one variable in {path}: {keys}. Pass variable_name explicitly.")
        variable_name = keys[0]

    arr = mat[variable_name]
    if transpose:
        arr = arr.T
    return arr


def scenario_label(scenario: str) -> int:
    return 0 if scenario.lower().startswith("clean") else 1


def make_time_bin(series: pd.Series, decimals: int = TIME_BIN_DECIMALS) -> pd.Series:
    return series.round(decimals)


def ecef_to_geodetic(x, y, z):
    """Convert ECEF XYZ meters to WGS84 latitude, longitude, height.

    Returns:
        lat_deg, lon_deg, height_m

    Uses iterative WGS84 conversion.
    """
    # WGS84 constants
    a = 6378137.0
    e2 = 6.69437999014e-3

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    z = np.asarray(z, dtype=float)

    lon = np.arctan2(y, x)
    p = np.sqrt(x**2 + y**2)
    lat = np.arctan2(z, p * (1 - e2))

    for _ in range(7):
        sin_lat = np.sin(lat)
        N = a / np.sqrt(1 - e2 * sin_lat**2)
        h = p / np.cos(lat) - N
        lat = np.arctan2(z, p * (1 - e2 * N / (N + h)))

    sin_lat = np.sin(lat)
    N = a / np.sqrt(1 - e2 * sin_lat**2)
    h = p / np.cos(lat) - N

    lat_deg = np.degrees(lat)
    lon_deg = np.degrees(lon)
    return lat_deg, lon_deg, h


def safe_entropy(power: np.ndarray, eps: float = 1e-12) -> float:
    p = np.asarray(power, dtype=float)
    total = np.sum(p)
    if total <= eps or len(p) <= 1:
        return np.nan
    prob = p / (total + eps)
    ent = -np.sum(prob * np.log(prob + eps))
    return ent / np.log(len(prob))


def spectral_features_from_complex(z: np.ndarray, min_samples: int = PSD_MIN_SAMPLES) -> dict:
    """Compute dynamic PSD-like features from high-rate IQ symaccumulation samples.

    This is not raw RF PSD. It is a compact spectrum of GRID iq.mat samples.
    """
    z = np.asarray(z, dtype=np.complex128)
    z = z[np.isfinite(z.real) & np.isfinite(z.imag)]

    if len(z) < min_samples:
        return {
            "iq_sample_count": len(z),
            "iq_power_mean": np.nan,
            "iq_power_std": np.nan,
            "iq_phase_std": np.nan,
            "psd_total_power": np.nan,
            "psd_peak_ratio": np.nan,
            "psd_spectral_entropy": np.nan,
            "psd_flatness": np.nan,
            "psd_low_high_ratio": np.nan,
        }

    mag2 = np.abs(z) ** 2
    phase = np.unwrap(np.angle(z))

    z0 = z - np.mean(z)
    spec = np.fft.fftshift(np.fft.fft(z0))
    psd = np.abs(spec) ** 2

    eps = 1e-12
    total_power = float(np.sum(psd))
    peak_power = float(np.max(psd))
    half = len(psd) // 2
    low = float(np.sum(psd[:half]))
    high = float(np.sum(psd[half:]))

    return {
        "iq_sample_count": len(z),
        "iq_power_mean": float(np.mean(mag2)),
        "iq_power_std": float(np.std(mag2)),
        "iq_phase_std": float(np.std(phase)),
        "psd_total_power": total_power,
        "psd_peak_ratio": peak_power / (total_power + eps),
        "psd_spectral_entropy": safe_entropy(psd),
        "psd_flatness": float(np.exp(np.mean(np.log(psd + eps))) / (np.mean(psd) + eps)),
        "psd_low_high_ratio": low / (high + eps),
    }


def impute_median_with_flag(df: pd.DataFrame, cols: list[str], flag_name: str) -> pd.DataFrame:
    """Add availability flag and median-impute selected columns."""
    df = df.copy()
    existing = [c for c in cols if c in df.columns]
    if not existing:
        df[flag_name] = 0
        for c in cols:
            df[c] = 0.0
        return df

    df[flag_name] = df[existing].notna().all(axis=1).astype(int)
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
        median = df[c].median(skipna=True)
        if pd.isna(median):
            median = 0.0
        df[c] = df[c].fillna(median)
    return df

## 3. File check

In [4]:
for scenario in SCENARIOS:
    d = BASE_DIR / scenario
    print("\nSCENARIO:", scenario, "exists:", d.exists())
    for fname in ["channel.mat", "navsol.mat", "iq.mat", "iqtaps.mat"]:
        p = d / fname
        if p.exists():
            print(f"  {fname:12s}", round(p.stat().st_size / 1024 / 1024, 2), "MB")
        else:
            print(f"  {fname:12s} MISSING")
    print("  power:", [p.name for p in sorted(d.glob("*power*MHz.mat"))])


SCENARIO: cleanStatic exists: True
  channel.mat  3.41 MB
  navsol.mat   0.19 MB
  iq.mat       84.02 MB
  iqtaps.mat   657.12 MB
  power: ['cd1power2MHz.mat', 'cd1power4MHz.mat', 'cd1power8MHz.mat']

SCENARIO: ds2 exists: True
  channel.mat  3.33 MB
  navsol.mat   0.19 MB
  iq.mat       82.87 MB
  iqtaps.mat   648.09 MB
  power: ['tb2power2MHz.mat', 'tb2power4MHz.mat', 'tb2power8MHz.mat']

SCENARIO: ds3 exists: True
  channel.mat  3.41 MB
  navsol.mat   0.19 MB
  iq.mat       83.98 MB
  iqtaps.mat   656.8 MB
  power: ['tb3power2MHz.mat', 'tb3power4MHz.mat', 'tb3power8MHz.mat']

SCENARIO: ds7 exists: True
  channel.mat  3.5 MB
  navsol.mat   0.2 MB
  iq.mat       86.24 MB
  iqtaps.mat   674.44 MB
  power: ['tb7power2MHz.mat', 'tb7power4MHz.mat', 'tb7power8MHz.mat']


## 4. Channel features: CN0, Doppler, PLL/DLL, stability

In [5]:
CHANNEL_COLUMNS = [
    "rrt_week", "rrt_seconds", "ort_week", "ort_whole_seconds", "ort_fractional_second",
    "doppler_hz", "beat_carrier_phase_cycles", "pseudorange_m", "cn0_dbhz",
    "measurement_valid", "error_indicator", "channel_status", "signal_type", "txid",
]


def extract_channel_features(scenario: str) -> pd.DataFrame:
    arr = load_grid_mat(BASE_DIR / scenario / "channel.mat", variable_name="channel", transpose=True)
    if arr.shape[1] != len(CHANNEL_COLUMNS):
        raise ValueError(f"{scenario}: expected 14 columns, got {arr.shape}")

    df = pd.DataFrame(arr, columns=CHANNEL_COLUMNS)
    df.insert(0, "scenario", scenario)

    for col in ["measurement_valid", "error_indicator", "channel_status", "signal_type", "txid"]:
        df[col] = df[col].astype(int)

    df["ort_valid"] = df["ort_week"] != 9999
    df["time_s"] = df["ort_whole_seconds"] + df["ort_fractional_second"]
    valid_time = df.loc[df["ort_valid"], "time_s"]
    start_time = valid_time.min() if len(valid_time) else df["rrt_seconds"].min()

    df["time_from_start_s"] = df["time_s"] - start_time
    df.loc[~df["ort_valid"], "time_from_start_s"] = df.loc[~df["ort_valid"], "rrt_seconds"] - df["rrt_seconds"].min()
    df["time_bin"] = make_time_bin(df["time_from_start_s"])

    df["label_spoofed_scenario"] = scenario_label(scenario)

    err = df["error_indicator"].fillna(0).astype(int)
    df["phase_tracking_anomaly"] = ((err & 0b001) > 0).astype(int)
    df["spoofing_detected_grid"] = ((err & 0b010) > 0).astype(int)
    df["possible_half_cycle_phase_offset"] = ((err & 0b100) > 0).astype(int)

    df["pll_lock"] = (df["channel_status"] >= 5).astype(int)
    df["dll_lock"] = ((df["measurement_valid"] == 1) & (df["channel_status"] >= 4)).astype(int)

    group_cols = ["txid", "signal_type"]
    df = df.sort_values(group_cols + ["time_from_start_s"]).reset_index(drop=True)
    df["cn0_delta"] = df.groupby(group_cols)["cn0_dbhz"].diff()
    df["doppler_delta"] = df.groupby(group_cols)["doppler_hz"].diff()
    df["pseudorange_delta"] = df.groupby(group_cols)["pseudorange_m"].diff()
    df["carrier_phase_delta"] = df.groupby(group_cols)["beat_carrier_phase_cycles"].diff()
    df["cn0_roll_std_5"] = df.groupby(group_cols)["cn0_dbhz"].rolling(5, min_periods=2).std().reset_index(level=group_cols, drop=True)
    df["doppler_roll_std_5"] = df.groupby(group_cols)["doppler_hz"].rolling(5, min_periods=2).std().reset_index(level=group_cols, drop=True)

    keep = [
        "scenario", "label_spoofed_scenario", "spoofing_detected_grid",
        "time_from_start_s", "time_bin", "txid", "signal_type",
        "cn0_dbhz", "doppler_hz", "pseudorange_m", "beat_carrier_phase_cycles",
        "pll_lock", "dll_lock",
        "cn0_delta", "doppler_delta", "pseudorange_delta", "carrier_phase_delta",
        "cn0_roll_std_5", "doppler_roll_std_5",
        "phase_tracking_anomaly", "possible_half_cycle_phase_offset",
    ]
    return df[keep].copy()

## 5. Navsol features: ECEF + latitude/longitude/height + position drift

In [6]:
NAVSOL_COLUMNS_13 = [
    "ort_week", "ort_whole_seconds", "ort_fractional_second",
    "ecef_x_m", "ecef_y_m", "ecef_z_m", "clock_error_m",
    "ecef_vx_mps", "ecef_vy_mps", "ecef_vz_mps", "clock_error_rate_mps",
    "solution_flag", "nis_ratio",
]
NAVSOL_COLUMNS_12 = NAVSOL_COLUMNS_13[:-1]


def extract_navsol_features(scenario: str) -> pd.DataFrame:
    path = BASE_DIR / scenario / "navsol.mat"
    if not path.exists():
        return pd.DataFrame()

    arr = load_grid_mat(path, variable_name="navsol", transpose=True)
    if arr.shape[1] == 13:
        cols = NAVSOL_COLUMNS_13
    elif arr.shape[1] == 12:
        cols = NAVSOL_COLUMNS_12
    else:
        raise ValueError(f"{scenario}: expected 12 or 13 navsol columns, got {arr.shape}")

    df = pd.DataFrame(arr, columns=cols)
    df.insert(0, "scenario", scenario)
    df["label_spoofed_scenario"] = scenario_label(scenario)
    df["time_s"] = df["ort_whole_seconds"] + df["ort_fractional_second"]
    df["time_from_start_s"] = df["time_s"] - df["time_s"].min()
    df["time_bin"] = make_time_bin(df["time_from_start_s"])

    # ECEF trajectory.
    xyz = df[["ecef_x_m", "ecef_y_m", "ecef_z_m"]].to_numpy()
    origin = xyz[0]
    df["distance_from_start_m"] = np.linalg.norm(xyz - origin, axis=1)
    df["position_delta_m"] = df["distance_from_start_m"].diff().fillna(0.0)

    # Convert ECEF to geodetic WGS84.
    lat, lon, height = ecef_to_geodetic(df["ecef_x_m"], df["ecef_y_m"], df["ecef_z_m"])
    df["latitude_deg"] = lat
    df["longitude_deg"] = lon
    df["height_m"] = height

    # Relative geolocation features.
    df["latitude_delta_deg"] = df["latitude_deg"] - df["latitude_deg"].iloc[0]
    df["longitude_delta_deg"] = df["longitude_deg"] - df["longitude_deg"].iloc[0]
    df["height_delta_m"] = df["height_m"] - df["height_m"].iloc[0]

    # Velocity norm.
    vel = df[["ecef_vx_mps", "ecef_vy_mps", "ecef_vz_mps"]].to_numpy()
    df["speed_mps"] = np.linalg.norm(vel, axis=1)

    if "nis_ratio" not in df.columns:
        df["nis_ratio"] = np.nan

    keep = [
        "scenario", "label_spoofed_scenario", "time_bin",

        # absolute geolocation / ECEF
        "ecef_x_m", "ecef_y_m", "ecef_z_m",
        "latitude_deg", "longitude_deg", "height_m",

        # relative geolocation
        "latitude_delta_deg", "longitude_delta_deg", "height_delta_m",
        "distance_from_start_m", "position_delta_m",

        # motion / clock / quality
        "speed_mps", "clock_error_m", "clock_error_rate_mps", "solution_flag", "nis_ratio",
    ]

    return (
        df[keep]
        .groupby(["scenario", "label_spoofed_scenario", "time_bin"], as_index=False)
        .agg(
            ecef_x_m=("ecef_x_m", "mean"),
            ecef_y_m=("ecef_y_m", "mean"),
            ecef_z_m=("ecef_z_m", "mean"),

            latitude_deg=("latitude_deg", "mean"),
            longitude_deg=("longitude_deg", "mean"),
            height_m=("height_m", "mean"),

            latitude_delta_deg=("latitude_delta_deg", "mean"),
            longitude_delta_deg=("longitude_delta_deg", "mean"),
            height_delta_m=("height_delta_m", "mean"),
            distance_from_start_m=("distance_from_start_m", "mean"),
            position_delta_m=("position_delta_m", "mean"),

            speed_mps=("speed_mps", "mean"),
            clock_error_m=("clock_error_m", "mean"),
            clock_error_rate_mps=("clock_error_rate_mps", "mean"),
            solution_flag=("solution_flag", "max"),
            nis_ratio=("nis_ratio", "mean"),
        )
    )

## 6. Correlator distortion from iqtaps.mat

In [7]:
def extract_correlator_features(scenario: str, max_rows: int | None = IQTAPS_MAX_ROWS) -> pd.DataFrame:
    path = BASE_DIR / scenario / "iqtaps.mat"
    if not path.exists():
        return pd.DataFrame()

    arr = load_grid_mat(path, variable_name="iqtaps", transpose=True)
    if max_rows is not None and arr.shape[0] > max_rows:
        arr = arr[:max_rows, :]

    total_cols = arr.shape[1]
    if total_cols < 6 or (total_cols - 4) % 2 != 0:
        raise ValueError(f"{scenario}: unexpected iqtaps shape {arr.shape}")

    n_taps = (total_cols - 4) // 2
    base = pd.DataFrame({
        "scenario": scenario,
        "rrt_seconds": arr[:, 1],
        "signal_type": arr[:, 2].astype(int),
        "txid": arr[:, 3].astype(int),
    })
    base["time_from_start_s"] = base["rrt_seconds"] - base["rrt_seconds"].min()
    base["time_bin"] = make_time_bin(base["time_from_start_s"])

    I = arr[:, 4:4+n_taps]
    Q = arr[:, 4+n_taps:4+2*n_taps]
    power = I**2 + Q**2

    eps = 1e-12
    peak_power = power.max(axis=1)
    total_power = power.sum(axis=1)
    center = n_taps // 2
    left_power = power[:, :center].sum(axis=1)
    right_power = power[:, center+1:].sum(axis=1)

    base["corr_peak_ratio"] = peak_power / (total_power + eps)
    base["corr_asymmetry"] = (right_power - left_power) / (right_power + left_power + eps)
    base["corr_width_proxy"] = (power >= (0.5 * peak_power[:, None])).sum(axis=1)
    base["corr_center_ratio"] = power[:, center] / (total_power + eps)

    return (
        base.groupby(["scenario", "txid", "signal_type", "time_bin"], as_index=False)
        .agg(
            corr_peak_ratio_mean=("corr_peak_ratio", "mean"),
            corr_asymmetry_mean=("corr_asymmetry", "mean"),
            corr_width_proxy_mean=("corr_width_proxy", "mean"),
            corr_center_ratio_mean=("corr_center_ratio", "mean"),
        )
    )

## 7. Dynamic PSD-like features from iq.mat

In [8]:
IQ_COLUMNS = [
    "rrt_week", "rrt_seconds", "ort_week", "ort_whole_seconds", "ort_fractional_second",
    "beat_carrier_phase_cycles", "i_symaccum", "q_symaccum", "data_symbol", "signal_type", "txid",
]


def extract_dynamic_iq_psd_features(scenario: str, max_rows: int | None = IQ_MAX_ROWS) -> pd.DataFrame:
    path = BASE_DIR / scenario / "iq.mat"
    if not path.exists():
        return pd.DataFrame()

    arr = load_grid_mat(path, variable_name="iq", transpose=True)
    if max_rows is not None and arr.shape[0] > max_rows:
        arr = arr[:max_rows, :]

    if arr.shape[1] != len(IQ_COLUMNS):
        raise ValueError(f"{scenario}: expected 11 iq columns, got {arr.shape}")

    df = pd.DataFrame(arr, columns=IQ_COLUMNS)
    df.insert(0, "scenario", scenario)
    df["signal_type"] = df["signal_type"].astype(int)
    df["txid"] = df["txid"].astype(int)

    df["ort_valid"] = df["ort_week"] != 9999
    df["time_s"] = df["ort_whole_seconds"] + df["ort_fractional_second"]
    valid_time = df.loc[df["ort_valid"], "time_s"]
    start_time = valid_time.min() if len(valid_time) else df["rrt_seconds"].min()
    df["time_from_start_s"] = df["time_s"] - start_time
    df.loc[~df["ort_valid"], "time_from_start_s"] = df.loc[~df["ort_valid"], "rrt_seconds"] - df["rrt_seconds"].min()
    df["time_bin"] = make_time_bin(df["time_from_start_s"])
    df["iq_complex"] = df["i_symaccum"].to_numpy() + 1j * df["q_symaccum"].to_numpy()

    rows = []
    for (sc, txid, sig, tb), g in df.groupby(["scenario", "txid", "signal_type", "time_bin"]):
        feats = spectral_features_from_complex(g["iq_complex"].to_numpy())
        feats.update({"scenario": sc, "txid": txid, "signal_type": sig, "time_bin": tb})
        rows.append(feats)

    return pd.DataFrame(rows)

## 8. Global power features from power*.mat

In [9]:
def extract_global_power_features(scenario: str) -> pd.DataFrame:
    result = {"scenario": scenario}
    power_files = sorted((BASE_DIR / scenario).glob("*power*MHz.mat"))

    for path in power_files:
        mat = sio.loadmat(path)
        keys = [k for k in mat.keys() if not k.startswith("__")]
        if not keys:
            continue

        values = np.asarray(mat[keys[0]]).ravel()
        name = path.name.lower()
        if "2mhz" in name:
            bw = "2mhz"
        elif "4mhz" in name:
            bw = "4mhz"
        elif "8mhz" in name:
            bw = "8mhz"
        else:
            bw = "unknown"

        result[f"global_power_{bw}_mean"] = np.nanmean(values)
        result[f"global_power_{bw}_std"] = np.nanstd(values)
        result[f"global_power_{bw}_median"] = np.nanmedian(values)

    return pd.DataFrame([result])

## 9. Build source feature tables

In [10]:
# Channel
channel_parts = []
for scenario in SCENARIOS:
    print("Loading channel:", scenario)
    part = extract_channel_features(scenario)
    print(" ", part.shape)
    channel_parts.append(part)
channel_features = pd.concat(channel_parts, ignore_index=True)
channel_features.to_csv(OUTPUT_DIR / "source_channel_features_v4.csv", index=False)
print("channel_features:", channel_features.shape)
display(channel_features.head())

Loading channel: cleanStatic
  (31892, 21)
Loading channel: ds2
  (31203, 21)
Loading channel: ds3
  (31892, 21)
Loading channel: ds7
  (32732, 21)
channel_features: (127719, 21)


,scenario,label_spoofed_scenario,spoofing_detected_grid,time_from_start_s,time_bin,txid,signal_type,cn0_dbhz,doppler_hz,pseudorange_m,beat_carrier_phase_cycles,pll_lock,dll_lock,cn0_delta,doppler_delta,pseudorange_delta,carrier_phase_delta,cn0_roll_std_5,doppler_roll_std_5,phase_tracking_anomaly,possible_half_cycle_phase_offset
0,cleanStatic,0,0,0.0,0.0,3,0,50.878071,780.029236,0.000000e+00,-0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,cleanStatic,0,0,0.0,0.0,3,0,50.934761,766.088501,2.098547e+07,-20767.900704,1,1,0.056690,-13.940735,2.098547e+07,-20767.900704,0.040086,9.857588,0,0
2,cleanStatic,0,0,0.2,0.0,3,0,50.903549,777.182861,0.000000e+00,-0.000000,0,0,-0.031212,11.094360,-2.098547e+07,20767.900704,0.028393,7.365808,0,0
3,cleanStatic,0,0,0.2,0.0,3,0,50.928856,765.873108,2.098544e+07,-20921.099800,1,1,0.025307,-11.309753,2.098544e+07,-20921.099800,0.025967,7.381755,0,0
4,cleanStatic,0,0,0.4,0.0,3,0,50.917503,779.016418,0.000000e+00,-0.000000,0,0,-0.011353,13.143311,-2.098544e+07,20921.099800,0.022658,7.064521,0,0


In [11]:
# Navsol with geolocation
navsol_parts = []
for scenario in SCENARIOS:
    print("Loading navsol/geolocation:", scenario)
    try:
        part = extract_navsol_features(scenario)
        print(" ", part.shape)
        if not part.empty:
            navsol_parts.append(part)
    except Exception as e:
        print(f"  skipped {scenario}: {type(e).__name__}: {e}")
navsol_features = pd.concat(navsol_parts, ignore_index=True) if navsol_parts else pd.DataFrame()
if not navsol_features.empty:
    navsol_features.to_csv(OUTPUT_DIR / "source_navsol_geolocation_features_v4.csv", index=False)
print("navsol_features:", navsol_features.shape)
display(navsol_features.head() if not navsol_features.empty else navsol_features)

Loading navsol/geolocation: cleanStatic
  (424, 19)
Loading navsol/geolocation: ds2
  (421, 19)
Loading navsol/geolocation: ds3
  (420, 19)
Loading navsol/geolocation: ds7
  (436, 19)
navsol_features: (1701, 19)


,scenario,label_spoofed_scenario,time_bin,ecef_x_m,ecef_y_m,ecef_z_m,latitude_deg,longitude_deg,height_m,latitude_delta_deg,longitude_delta_deg,height_delta_m,distance_from_start_m,position_delta_m,speed_mps,clock_error_m,clock_error_rate_mps,solution_flag,nis_ratio
0,cleanStatic,0,0.0,-741978.144298,-5.462175e+06,3.198011e+06,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.218500,-0.000374,2.0,NaN
1,cleanStatic,0,1.0,-741983.853677,-5.462197e+06,3.198014e+06,30.287614,-97.735699,135.102927,-0.000105,-0.000029,34.167272,36.284631,4.166498,0.0,70820.639257,0.002077,2.0,NaN
2,cleanStatic,0,2.0,-741984.850845,-5.462208e+06,3.198016e+06,30.287583,-97.735695,145.301223,-0.000136,-0.000025,44.365569,46.940929,1.114204,0.0,70825.325478,0.003356,2.0,NaN
3,cleanStatic,0,3.0,-741986.374133,-5.462212e+06,3.198017e+06,30.287567,-97.735704,149.835538,-0.000152,-0.000034,48.899883,51.842799,0.527155,0.0,70827.368567,0.003332,2.0,NaN
4,cleanStatic,0,4.0,-741986.199171,-5.462213e+06,3.198019e+06,30.287584,-97.735702,151.031269,-0.000136,-0.000032,50.095615,52.398270,0.116782,0.0,70827.828483,0.002981,2.0,NaN


In [12]:
# Correlator
if LOAD_CORRELATOR_FEATURES:
    corr_parts = []
    for scenario in SCENARIOS:
        print("Loading iqtaps/correlator:", scenario)
        try:
            part = extract_correlator_features(scenario)
            print(" ", part.shape)
            if not part.empty:
                corr_parts.append(part)
        except Exception as e:
            print(f"  skipped {scenario}: {type(e).__name__}: {e}")
    correlator_features = pd.concat(corr_parts, ignore_index=True) if corr_parts else pd.DataFrame()
else:
    correlator_features = pd.DataFrame()

if not correlator_features.empty:
    correlator_features.to_csv(OUTPUT_DIR / "source_correlator_features_v4.csv", index=False)
print("correlator_features:", correlator_features.shape)
display(correlator_features.head() if not correlator_features.empty else correlator_features)

Loading iqtaps/correlator: cleanStatic
  (1918, 8)
Loading iqtaps/correlator: ds2
  (1918, 8)
Loading iqtaps/correlator: ds3
  (1931, 8)
Loading iqtaps/correlator: ds7
  (1918, 8)
correlator_features: (7685, 8)


,scenario,txid,signal_type,time_bin,corr_peak_ratio_mean,corr_asymmetry_mean,corr_width_proxy_mean,corr_center_ratio_mean
0,cleanStatic,3,0,0.0,0.073815,0.004171,11.965517,0.073815
1,cleanStatic,3,0,1.0,0.074040,0.010213,11.770000,0.074040
2,cleanStatic,3,0,2.0,0.073960,0.011461,11.960000,0.073960
3,cleanStatic,3,0,3.0,0.073850,0.009769,11.880000,0.073850
4,cleanStatic,3,0,4.0,0.073842,0.006980,11.860000,0.073842


In [13]:
# Dynamic IQ PSD
if LOAD_DYNAMIC_IQ_PSD_FEATURES:
    iq_parts = []
    for scenario in SCENARIOS:
        print("Loading iq/dynamic PSD:", scenario)
        try:
            part = extract_dynamic_iq_psd_features(scenario)
            print(" ", part.shape)
            if not part.empty:
                iq_parts.append(part)
        except Exception as e:
            print(f"  skipped {scenario}: {type(e).__name__}: {e}")
    iq_psd_features = pd.concat(iq_parts, ignore_index=True) if iq_parts else pd.DataFrame()
else:
    iq_psd_features = pd.DataFrame()

if not iq_psd_features.empty:
    iq_psd_features.to_csv(OUTPUT_DIR / "source_dynamic_iq_psd_features_v4.csv", index=False)
print("iq_psd_features:", iq_psd_features.shape)
display(iq_psd_features.head() if not iq_psd_features.empty else iq_psd_features)

Loading iq/dynamic PSD: cleanStatic
  (6020, 13)
Loading iq/dynamic PSD: ds2
  (5924, 13)
Loading iq/dynamic PSD: ds3
  (6048, 13)
Loading iq/dynamic PSD: ds7
  (6188, 13)
iq_psd_features: (24180, 13)


,iq_sample_count,iq_power_mean,iq_power_std,iq_phase_std,psd_total_power,psd_peak_ratio,psd_spectral_entropy,psd_flatness,psd_low_high_ratio,scenario,txid,signal_type,time_bin
0,79,5.280769e+10,2.733417e+09,0.497484,7.267695e+13,0.249684,0.638407,0.122613,1.300054,cleanStatic,3,0,0.0
1,200,5.311448e+10,2.059342e+09,0.514856,5.040232e+14,0.278084,0.408611,0.018127,1.053578,cleanStatic,3,0,1.0
2,200,5.355861e+10,2.109831e+09,0.020919,1.764996e+12,0.047171,0.904445,0.418595,1.197715,cleanStatic,3,0,2.0
3,200,5.306631e+10,2.094258e+09,0.023706,2.018148e+12,0.023295,0.922605,0.435968,1.559673,cleanStatic,3,0,3.0
4,200,5.286041e+10,2.550785e+09,0.026187,2.681141e+12,0.040502,0.899063,0.410293,1.014458,cleanStatic,3,0,4.0


In [14]:
# Global power
global_power_features = pd.concat([extract_global_power_features(s) for s in SCENARIOS], ignore_index=True)
global_power_features.to_csv(OUTPUT_DIR / "source_global_power_features_v4.csv", index=False)
print("global_power_features:", global_power_features.shape)
display(global_power_features)

global_power_features: (4, 10)


,scenario,global_power_2mhz_mean,global_power_2mhz_std,global_power_2mhz_median,global_power_4mhz_mean,global_power_4mhz_std,global_power_4mhz_median,global_power_8mhz_mean,global_power_8mhz_std,global_power_8mhz_median
0,cleanStatic,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9
1,ds2,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9
2,ds3,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9
3,ds7,233.9,135.099951,233.9,233.9,135.099951,233.9,233.9,135.099951,233.9


## 10. Event-level merged features

In [15]:
event_features = channel_features.copy()

if not correlator_features.empty:
    event_features = event_features.merge(
        correlator_features,
        on=["scenario", "txid", "signal_type", "time_bin"],
        how="left",
    )

if not iq_psd_features.empty:
    event_features = event_features.merge(
        iq_psd_features,
        on=["scenario", "txid", "signal_type", "time_bin"],
        how="left",
    )

if not navsol_features.empty:
    event_features = event_features.merge(
        navsol_features.drop(columns=["label_spoofed_scenario"], errors="ignore"),
        on=["scenario", "time_bin"],
        how="left",
    )

if KEEP_GLOBAL_POWER_FOR_ANALYSIS:
    event_features = event_features.merge(global_power_features, on="scenario", how="left")

# Missing handling with availability flags.
corr_cols = ["corr_peak_ratio_mean", "corr_asymmetry_mean", "corr_width_proxy_mean", "corr_center_ratio_mean"]
iq_cols = ["iq_power_mean", "iq_power_std", "iq_phase_std", "psd_total_power", "psd_peak_ratio", "psd_spectral_entropy", "psd_flatness", "psd_low_high_ratio"]
nav_cols = [
    "ecef_x_m", "ecef_y_m", "ecef_z_m",
    "latitude_deg", "longitude_deg", "height_m",
    "latitude_delta_deg", "longitude_delta_deg", "height_delta_m",
    "distance_from_start_m", "position_delta_m",
    "speed_mps", "clock_error_m", "clock_error_rate_mps", "solution_flag", "nis_ratio",
]

event_features = impute_median_with_flag(event_features, corr_cols, "corr_available")
event_features = impute_median_with_flag(event_features, iq_cols, "iq_psd_available")
event_features = impute_median_with_flag(event_features, nav_cols, "navsol_available")

# Basic deltas/rolling NaN imputation.
basic_nan_cols = ["cn0_delta", "doppler_delta", "pseudorange_delta", "carrier_phase_delta", "cn0_roll_std_5", "doppler_roll_std_5"]
for c in basic_nan_cols:
    if c in event_features.columns:
        event_features[c] = event_features[c].fillna(0.0)

event_features.to_csv(OUTPUT_DIR / "event_features_v4.csv", index=False)
print("event_features:", event_features.shape)
display(event_features.head())

/home/aw/envs/ml310/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


event_features: (127719, 62)


,scenario,label_spoofed_scenario,spoofing_detected_grid,time_from_start_s,time_bin,txid,signal_type,cn0_dbhz,doppler_hz,pseudorange_m,beat_carrier_phase_cycles,pll_lock,dll_lock,cn0_delta,doppler_delta,pseudorange_delta,carrier_phase_delta,cn0_roll_std_5,doppler_roll_std_5,phase_tracking_anomaly,possible_half_cycle_phase_offset,corr_peak_ratio_mean,corr_asymmetry_mean,corr_width_proxy_mean,corr_center_ratio_mean,iq_sample_count,iq_power_mean,iq_power_std,iq_phase_std,psd_total_power,psd_peak_ratio,psd_spectral_entropy,psd_flatness,psd_low_high_ratio,ecef_x_m,ecef_y_m,ecef_z_m,latitude_deg,longitude_deg,height_m,latitude_delta_deg,longitude_delta_deg,height_delta_m,distance_from_start_m,position_delta_m,speed_mps,clock_error_m,clock_error_rate_mps,solution_flag,nis_ratio,global_power_2mhz_mean,global_power_2mhz_std,global_power_2mhz_median,global_power_4mhz_mean,global_power_4mhz_std,global_power_4mhz_median,global_power_8mhz_mean,global_power_8mhz_std,global_power_8mhz_median,corr_available,iq_psd_available,navsol_available
0,cleanStatic,0,0,0.0,0.0,3,0,50.878071,780.029236,0.000000e+00,-0.000000,0,0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0,0,0.073815,0.004171,11.965517,0.073815,79,5.280769e+10,2.733417e+09,0.497484,7.267695e+13,0.249684,0.638407,0.122613,1.300054,-741978.144298,-5.462175e+06,3.198011e+06,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9,1,1,0
1,cleanStatic,0,0,0.0,0.0,3,0,50.934761,766.088501,2.098547e+07,-20767.900704,1,1,0.056690,-13.940735,2.098547e+07,-20767.900704,0.040086,9.857588,0,0,0.073815,0.004171,11.965517,0.073815,79,5.280769e+10,2.733417e+09,0.497484,7.267695e+13,0.249684,0.638407,0.122613,1.300054,-741978.144298,-5.462175e+06,3.198011e+06,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9,1,1,0
2,cleanStatic,0,0,0.2,0.0,3,0,50.903549,777.182861,0.000000e+00,-0.000000,0,0,-0.031212,11.094360,-2.098547e+07,20767.900704,0.028393,7.365808,0,0,0.073815,0.004171,11.965517,0.073815,79,5.280769e+10,2.733417e+09,0.497484,7.267695e+13,0.249684,0.638407,0.122613,1.300054,-741978.144298,-5.462175e+06,3.198011e+06,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9,1,1,0
3,cleanStatic,0,0,0.2,0.0,3,0,50.928856,765.873108,2.098544e+07,-20921.099800,1,1,0.025307,-11.309753,2.098544e+07,-20921.099800,0.025967,7.381755,0,0,0.073815,0.004171,11.965517,0.073815,79,5.280769e+10,2.733417e+09,0.497484,7.267695e+13,0.249684,0.638407,0.122613,1.300054,-741978.144298,-5.462175e+06,3.198011e+06,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9,1,1,0
4,cleanStatic,0,0,0.4,0.0,3,0,50.917503,779.016418,0.000000e+00,-0.000000,0,0,-0.011353,13.143311,-2.098544e+07,20921.099800,0.022658,7.064521,0,0,0.073815,0.004171,11.965517,0.073815,79,5.280769e+10,2.733417e+09,0.497484,7.267695e+13,0.249684,0.638407,0.122613,1.300054,-741978.144298,-5.462175e+06,3.198011e+06,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0,227.9,131.635849,227.9,227.9,131.635849,227.9,227.9,131.635849,227.9,1,1,0


## 11. Satellite-second aggregated features

In [16]:
satellite_second_features = (
    event_features
    .groupby(["scenario", "label_spoofed_scenario", "time_bin", "txid", "signal_type"], as_index=False)
    .agg(
        rows_in_bin=("scenario", "size"),
        grid_spoof_rate=("spoofing_detected_grid", "mean"),

        cn0_mean=("cn0_dbhz", "mean"),
        cn0_std=("cn0_dbhz", "std"),
        cn0_min=("cn0_dbhz", "min"),
        cn0_max=("cn0_dbhz", "max"),
        doppler_mean=("doppler_hz", "mean"),
        doppler_std=("doppler_hz", "std"),
        doppler_abs_mean=("doppler_hz", lambda x: np.mean(np.abs(x))),

        pll_lock_rate=("pll_lock", "mean"),
        dll_lock_rate=("dll_lock", "mean"),

        cn0_delta_mean=("cn0_delta", "mean"),
        doppler_delta_mean=("doppler_delta", "mean"),
        pseudorange_delta_mean=("pseudorange_delta", "mean"),
        carrier_phase_delta_mean=("carrier_phase_delta", "mean"),
        cn0_roll_std_5_mean=("cn0_roll_std_5", "mean"),
        doppler_roll_std_5_mean=("doppler_roll_std_5", "mean"),

        corr_available=("corr_available", "max"),
        corr_peak_ratio_mean=("corr_peak_ratio_mean", "mean"),
        corr_asymmetry_mean=("corr_asymmetry_mean", "mean"),
        corr_width_proxy_mean=("corr_width_proxy_mean", "mean"),
        corr_center_ratio_mean=("corr_center_ratio_mean", "mean"),

        iq_psd_available=("iq_psd_available", "max"),
        iq_power_mean=("iq_power_mean", "mean"),
        iq_power_std=("iq_power_std", "mean"),
        iq_phase_std=("iq_phase_std", "mean"),
        psd_peak_ratio=("psd_peak_ratio", "mean"),
        psd_spectral_entropy=("psd_spectral_entropy", "mean"),
        psd_flatness=("psd_flatness", "mean"),
        psd_low_high_ratio=("psd_low_high_ratio", "mean"),

        navsol_available=("navsol_available", "max"),
        latitude_deg=("latitude_deg", "mean"),
        longitude_deg=("longitude_deg", "mean"),
        height_m=("height_m", "mean"),
        latitude_delta_deg=("latitude_delta_deg", "mean"),
        longitude_delta_deg=("longitude_delta_deg", "mean"),
        height_delta_m=("height_delta_m", "mean"),
        distance_from_start_m=("distance_from_start_m", "mean"),
        position_delta_m=("position_delta_m", "mean"),
        speed_mps=("speed_mps", "mean"),
        clock_error_m=("clock_error_m", "mean"),
        clock_error_rate_mps=("clock_error_rate_mps", "mean"),
        solution_flag=("solution_flag", "max"),
        nis_ratio=("nis_ratio", "mean"),
    )
)

satellite_second_features = satellite_second_features.fillna(0.0)
satellite_second_features.to_csv(OUTPUT_DIR / "satellite_second_features_v4.csv", index=False)
print("satellite_second_features:", satellite_second_features.shape)
display(satellite_second_features.head())

satellite_second_features: (24180, 49)


,scenario,label_spoofed_scenario,time_bin,txid,signal_type,rows_in_bin,grid_spoof_rate,cn0_mean,cn0_std,cn0_min,cn0_max,doppler_mean,doppler_std,doppler_abs_mean,pll_lock_rate,dll_lock_rate,cn0_delta_mean,doppler_delta_mean,pseudorange_delta_mean,carrier_phase_delta_mean,cn0_roll_std_5_mean,doppler_roll_std_5_mean,corr_available,corr_peak_ratio_mean,corr_asymmetry_mean,corr_width_proxy_mean,corr_center_ratio_mean,iq_psd_available,iq_power_mean,iq_power_std,iq_phase_std,psd_peak_ratio,psd_spectral_entropy,psd_flatness,psd_low_high_ratio,navsol_available,latitude_deg,longitude_deg,height_m,latitude_delta_deg,longitude_delta_deg,height_delta_m,distance_from_start_m,position_delta_m,speed_mps,clock_error_m,clock_error_rate_mps,solution_flag,nis_ratio
0,cleanStatic,0,0.0,3,0,6,0.0,50.915383,0.021423,50.878071,50.934761,772.325439,7.089658,772.325439,0.5,0.5,0.008581,-2.377787,3.497569e+06,-3512.377204,0.021599,6.396617,1,0.073815,0.004171,11.965517,0.073815,1,5.280769e+10,2.733417e+09,0.497484,0.249684,0.638407,0.122613,1.300054,0,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0
1,cleanStatic,0,0.0,6,0,6,0.0,48.575165,0.153332,48.431114,48.720993,-516.484334,8.969645,516.484334,0.5,0.5,0.045268,-3.567154,3.652199e+06,2332.570137,0.137501,8.571764,1,0.072959,0.004284,12.142857,0.072959,1,3.027564e+10,1.833457e+09,0.784983,0.341593,0.584929,0.083003,0.296399,0,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0
2,cleanStatic,0,0.0,7,0,6,0.0,48.506321,0.174242,48.316288,48.678989,1854.850179,9.873736,1854.850179,0.5,0.5,0.056150,0.719828,3.780856e+06,-8402.959707,0.165833,5.240255,1,0.067873,-0.022835,13.000000,0.067873,1,3.063894e+10,1.970771e+09,0.157608,0.278349,0.660559,0.145010,1.471375,0,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0
3,cleanStatic,0,0.0,10,0,6,0.0,43.741171,0.566184,43.181297,44.276131,646.873383,15.583281,646.873383,0.5,0.5,-0.182472,-5.048543,3.988669e+06,-2906.372776,0.498793,13.888364,1,0.071203,-0.001861,12.458333,0.071203,1,9.373753e+09,1.964074e+09,0.366906,0.102203,0.831798,0.223264,2.514009,0,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0
4,cleanStatic,0,0.0,11,0,6,0.0,36.771917,1.979886,34.861649,38.610813,1917.998698,8.218965,1917.998698,0.5,0.5,-0.613141,-1.210714,4.339410e+06,-8669.479772,1.736369,6.845949,1,0.074099,0.080874,11.714286,0.074010,1,2.124037e+09,1.133354e+09,0.803826,0.199656,0.701799,0.155592,0.539619,0,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.2185,-0.000374,2.0,0.0


## 12. Global receiver-state features per second

In [17]:
global_time_features = (
    event_features
    .groupby(["scenario", "label_spoofed_scenario", "time_bin"], as_index=False)
    .agg(
        rows_in_bin=("scenario", "size"),
        satellite_count=("txid", "nunique"),
        signal_type_count=("signal_type", "nunique"),
        grid_spoof_rate=("spoofing_detected_grid", "mean"),

        cn0_mean_all=("cn0_dbhz", "mean"),
        cn0_std_all=("cn0_dbhz", "std"),
        cn0_min_all=("cn0_dbhz", "min"),
        cn0_max_all=("cn0_dbhz", "max"),
        doppler_mean_all=("doppler_hz", "mean"),
        doppler_std_all=("doppler_hz", "std"),
        doppler_abs_mean_all=("doppler_hz", lambda x: np.mean(np.abs(x))),

        pll_lock_rate_all=("pll_lock", "mean"),
        dll_lock_rate_all=("dll_lock", "mean"),
        phase_tracking_anomaly_rate_all=("phase_tracking_anomaly", "mean"),
        half_cycle_offset_rate_all=("possible_half_cycle_phase_offset", "mean"),

        cn0_delta_mean_all=("cn0_delta", "mean"),
        doppler_delta_mean_all=("doppler_delta", "mean"),
        cn0_roll_std_5_mean_all=("cn0_roll_std_5", "mean"),
        doppler_roll_std_5_mean_all=("doppler_roll_std_5", "mean"),

        corr_available_rate=("corr_available", "mean"),
        corr_peak_ratio_mean_all=("corr_peak_ratio_mean", "mean"),
        corr_asymmetry_mean_all=("corr_asymmetry_mean", "mean"),
        corr_width_proxy_mean_all=("corr_width_proxy_mean", "mean"),
        corr_center_ratio_mean_all=("corr_center_ratio_mean", "mean"),

        iq_psd_available_rate=("iq_psd_available", "mean"),
        iq_power_mean_all=("iq_power_mean", "mean"),
        iq_power_std_all=("iq_power_std", "mean"),
        iq_phase_std_all=("iq_phase_std", "mean"),
        psd_peak_ratio_mean_all=("psd_peak_ratio", "mean"),
        psd_spectral_entropy_mean_all=("psd_spectral_entropy", "mean"),
        psd_flatness_mean_all=("psd_flatness", "mean"),
        psd_low_high_ratio_mean_all=("psd_low_high_ratio", "mean"),

        navsol_available=("navsol_available", "max"),
        latitude_deg=("latitude_deg", "mean"),
        longitude_deg=("longitude_deg", "mean"),
        height_m=("height_m", "mean"),
        latitude_delta_deg=("latitude_delta_deg", "mean"),
        longitude_delta_deg=("longitude_delta_deg", "mean"),
        height_delta_m=("height_delta_m", "mean"),
        distance_from_start_m=("distance_from_start_m", "mean"),
        position_delta_m=("position_delta_m", "mean"),
        speed_mps=("speed_mps", "mean"),
        clock_error_m=("clock_error_m", "mean"),
        clock_error_rate_mps=("clock_error_rate_mps", "mean"),
        solution_flag=("solution_flag", "max"),
        nis_ratio=("nis_ratio", "mean"),
    )
)

global_time_features = global_time_features.fillna(0.0)
global_time_features.to_csv(OUTPUT_DIR / "global_time_features_v4.csv", index=False)
print("global_time_features:", global_time_features.shape)
display(global_time_features.head())

global_time_features: (1737, 49)


,scenario,label_spoofed_scenario,time_bin,rows_in_bin,satellite_count,signal_type_count,grid_spoof_rate,cn0_mean_all,cn0_std_all,cn0_min_all,cn0_max_all,doppler_mean_all,doppler_std_all,doppler_abs_mean_all,pll_lock_rate_all,dll_lock_rate_all,phase_tracking_anomaly_rate_all,half_cycle_offset_rate_all,cn0_delta_mean_all,doppler_delta_mean_all,cn0_roll_std_5_mean_all,doppler_roll_std_5_mean_all,corr_available_rate,corr_peak_ratio_mean_all,corr_asymmetry_mean_all,corr_width_proxy_mean_all,corr_center_ratio_mean_all,iq_psd_available_rate,iq_power_mean_all,iq_power_std_all,iq_phase_std_all,psd_peak_ratio_mean_all,psd_spectral_entropy_mean_all,psd_flatness_mean_all,psd_low_high_ratio_mean_all,navsol_available,latitude_deg,longitude_deg,height_m,latitude_delta_deg,longitude_delta_deg,height_delta_m,distance_from_start_m,position_delta_m,speed_mps,clock_error_m,clock_error_rate_mps,solution_flag,nis_ratio
0,cleanStatic,0,0.0,84,14,2,0.0,46.778046,4.811095,34.861649,52.710144,-246.376946,1996.261758,1632.716020,0.500000,0.5,0.0,0.0,-0.030225,-1.419161,0.311536,12.430748,1.0,0.072426,-0.000695,12.133391,0.072404,1.0,2.550778e+10,1.943180e+09,0.686820,0.253494,0.637723,0.142074,1.339948,0,30.287696,-97.735672,113.622687,-0.000024,-0.000002,12.687032,12.986138,7.647444,0.0,70813.218500,-0.000374,2.0,0.0
1,cleanStatic,0,1.0,140,14,2,0.0,46.823018,4.819249,34.774620,52.682785,-247.581791,1990.042900,1626.395885,0.571429,0.5,0.0,0.0,0.001037,-0.016339,0.331350,7.900230,1.0,0.072267,0.002913,12.166143,0.072201,1.0,2.556071e+10,1.926455e+09,0.445700,0.206412,0.539888,0.082134,1.011584,0,30.287614,-97.735699,135.102927,-0.000105,-0.000029,34.167272,36.284631,4.166498,0.0,70820.639257,0.002077,2.0,0.0
2,cleanStatic,0,2.0,140,14,2,0.0,46.840548,4.817704,34.912270,52.638035,-247.634057,1989.655631,1625.321746,0.992857,0.5,0.0,0.0,0.001411,-0.027108,0.324131,4.375614,1.0,0.072383,0.000657,12.186571,0.072267,1.0,2.554123e+10,1.966884e+09,0.066833,0.126173,0.787259,0.306570,1.025471,0,30.287583,-97.735695,145.301223,-0.000136,-0.000025,44.365569,46.940929,1.114204,0.0,70825.325478,0.003356,2.0,0.0
3,cleanStatic,0,3.0,140,14,2,0.0,46.830599,4.826641,34.938324,52.609219,-247.817700,1989.619491,1625.361785,1.000000,0.5,0.0,0.0,0.000550,-0.009273,0.337812,4.325797,1.0,0.072302,-0.005184,12.173571,0.072203,1.0,2.542938e+10,1.953786e+09,0.067720,0.113722,0.803480,0.320750,1.101752,0,30.287567,-97.735704,149.835538,-0.000152,-0.000034,48.899883,51.842799,0.527155,0.0,70827.368567,0.003332,2.0,0.0
4,cleanStatic,0,4.0,140,14,2,0.0,46.816712,4.835557,35.021271,52.552948,-248.088200,1989.555954,1625.328527,1.000000,0.5,0.0,0.0,-0.003016,-0.034674,0.338795,4.338718,1.0,0.072278,-0.002447,12.167143,0.072203,1.0,2.532932e+10,1.988495e+09,0.066453,0.100791,0.811819,0.328039,1.016090,0,30.287584,-97.735702,151.031269,-0.000136,-0.000032,50.095615,52.398270,0.116782,0.0,70827.828483,0.002981,2.0,0.0


## 13. Model-ready versions

In [18]:
# These are recommended model-ready files.
# We keep geolocation relative features. Absolute lat/lon are included too,
# but you can drop them before training to reduce scenario/location leakage.

model_ready_satellite = satellite_second_features.copy()
model_ready_global = global_time_features.copy()

# Also create a safer version without absolute lat/lon/ECEF.
absolute_geo_cols = [
    "latitude_deg", "longitude_deg", "height_m",
    "ecef_x_m", "ecef_y_m", "ecef_z_m",
]

model_ready_satellite_relative_geo = model_ready_satellite.drop(columns=absolute_geo_cols, errors="ignore")
model_ready_global_relative_geo = model_ready_global.drop(columns=absolute_geo_cols, errors="ignore")

model_ready_satellite.to_csv(OUTPUT_DIR / "model_ready_satellite_second_features_v4.csv", index=False)
model_ready_global.to_csv(OUTPUT_DIR / "model_ready_global_time_features_v4.csv", index=False)
model_ready_satellite_relative_geo.to_csv(OUTPUT_DIR / "model_ready_satellite_second_features_relative_geo_v4.csv", index=False)
model_ready_global_relative_geo.to_csv(OUTPUT_DIR / "model_ready_global_time_features_relative_geo_v4.csv", index=False)

print("Saved model-ready files:")
print(" -", OUTPUT_DIR / "model_ready_satellite_second_features_v4.csv")
print(" -", OUTPUT_DIR / "model_ready_global_time_features_v4.csv")
print(" -", OUTPUT_DIR / "model_ready_satellite_second_features_relative_geo_v4.csv")
print(" -", OUTPUT_DIR / "model_ready_global_time_features_relative_geo_v4.csv")

Saved model-ready files:
 - texbat_universal_features_v4/model_ready_satellite_second_features_v4.csv
 - texbat_universal_features_v4/model_ready_global_time_features_v4.csv
 - texbat_universal_features_v4/model_ready_satellite_second_features_relative_geo_v4.csv
 - texbat_universal_features_v4/model_ready_global_time_features_relative_geo_v4.csv


## 14. Sanity checks

In [19]:
def quality_report(df: pd.DataFrame, name: str):
    q = pd.DataFrame({
        "column": df.columns,
        "missing_rate": [df[c].isna().mean() for c in df.columns],
        "unique_count": [df[c].nunique(dropna=True) for c in df.columns],
    }).sort_values(["missing_rate", "unique_count"], ascending=[False, True])
    q.to_csv(OUTPUT_DIR / f"quality_{name}.csv", index=False)
    print("\nQUALITY:", name)
    display(q)


def scenario_summary(df: pd.DataFrame, name: str):
    summary = df.groupby("scenario").agg(
        rows=("scenario", "size"),
        label_mean=("label_spoofed_scenario", "mean"),
    ).reset_index()
    summary.to_csv(OUTPUT_DIR / f"summary_{name}.csv", index=False)
    print("\nSUMMARY:", name)
    display(summary)


quality_report(event_features, "event_features_v4")
quality_report(model_ready_satellite, "model_ready_satellite_second_features_v4")
quality_report(model_ready_global, "model_ready_global_time_features_v4")
quality_report(model_ready_global_relative_geo, "model_ready_global_time_features_relative_geo_v4")

scenario_summary(event_features, "event_features_v4")
scenario_summary(model_ready_satellite, "model_ready_satellite_second_features_v4")
scenario_summary(model_ready_global, "model_ready_global_time_features_v4")


QUALITY: event_features_v4


,column,missing_rate,unique_count
2,spoofing_detected_grid,0.0,1
20,possible_half_cycle_phase_offset,0.0,1
45,speed_mps,0.0,1
48,solution_flag,0.0,1
49,nis_ratio,0.0,1
...,...,...,...
7,cn0_dbhz,0.0,116990
18,doppler_roll_std_5,0.0,119384
15,pseudorange_delta,0.0,119886
16,carrier_phase_delta,0.0,119892



QUALITY: model_ready_satellite_second_features_v4


,column,missing_rate,unique_count
6,grid_spoof_rate,0.0,1
27,iq_psd_available,0.0,1
35,navsol_available,0.0,1
44,speed_mps,0.0,1
47,solution_flag,0.0,1
48,nis_ratio,0.0,1
1,label_spoofed_scenario,0.0,2
4,signal_type,0.0,2
22,corr_available,0.0,2
0,scenario,0.0,4



QUALITY: model_ready_global_time_features_v4


...

,scenario,rows,label_mean
0,cleanStatic,31892,0.0
1,ds2,31203,1.0
2,ds3,31892,1.0
3,ds7,32732,1.0



SUMMARY: model_ready_satellite_second_features_v4


,scenario,rows,label_mean
0,cleanStatic,6020,0.0
1,ds2,5924,1.0
2,ds3,6048,1.0
3,ds7,6188,1.0



SUMMARY: model_ready_global_time_features_v4


,scenario,rows,label_mean
0,cleanStatic,430,0.0
1,ds2,433,1.0
2,ds3,432,1.0
3,ds7,442,1.0


## 15. Suggested model usage

### Recommended first CSV

Use this one first:

```python
df = pd.read_csv(OUTPUT_DIR / 'model_ready_global_time_features_relative_geo_v4.csv')
```

This file keeps relative geolocation features but removes absolute `latitude_deg`, `longitude_deg`, `height_m`, and ECEF coordinates.

Target:

```python
y = df['label_spoofed_scenario']
```

Features:

```python
X = df.drop(columns=['scenario', 'label_spoofed_scenario', 'grid_spoof_rate'])
```

### For visualization

Use:

```python
geo_df = pd.read_csv(OUTPUT_DIR / 'model_ready_global_time_features_v4.csv')
```

Then plot:

- `latitude_deg` vs `longitude_deg`
- `distance_from_start_m` over `time_bin`
- `height_delta_m` over `time_bin`
- `clock_error_m` over `time_bin`

### Validation advice

Avoid random row split as the only metric. Prefer scenario-based split:

```python
train: cleanStatic + ds2 + ds3
test:  cleanStatic + ds7
```

This checks whether the model generalizes to a harder spoofing scenario.